# Web Scraping with Gaffa +  OpenAI

In this notebook, we'll use Gaffa to scrape a Wikipedia page and convert it to Markdown, then use gpt-4o to answer questions about the content.

## Step 1: Install Dependencies

Install the required libraries if you haven't already.

In [71]:
!pip install requests openai

## Step 2: Set Up API Keys

We'll load our Gaffa and OpenAI API keys from environment variables. You can get your Gaffa key from the dashboard at https://gaffa.dev/dashboard/api-keys and your OpenAI key from https://platform.openai.com/api-keys

## *If you are using Google Colab: Run the cell below, then skip to Step 3.*

In [77]:
import os
import requests
from openai import OpenAI
from google.colab import userdata

GAFFA_API_KEY = userdata.get("GAFFA_API_KEY")
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

if not GAFFA_API_KEY:
    raise RuntimeError("GAFFA_API_KEY is not set.")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY is not set.")

print("✅ API keys loaded successfully.")

✅ API keys loaded successfully.


## *If you are using an IDE and already have a `.env` file: Run the next two cells, then continue to Step 3.*

In [ ]:
# !pip install python-dotenv

In [ ]:
# import os
# import requests
# from openai import OpenAI
# from dotenv import load_dotenv

# load_dotenv()
# GAFFA_API_KEY = os.getenv("GAFFA_API_KEY")
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# if not GAFFA_API_KEY:
#     raise RuntimeError("GAFFA_API_KEY is not set.")
# if not OPENAI_API_KEY:
#     raise RuntimeError("OPENAI_API_KEY is not set.")

# print("✅ API keys loaded successfully.")

## Step 3: Scrape the Page with Gaffa

We send a request to Gaffa with the URL we want to scrape. Gaffa spins up a real browser, waits for the page to load, and converts the content to clean Markdown — handling proxies and anti-bot detection automatically.

In [73]:
def fetch_markdown_with_gaffa(url):
    payload = {
        "url": url,
        "proxy_location": None,
        "async": False,
        "max_cache_age": 0,
        "settings": {
            "record_request": False,
            "actions": [
                {
                    "type": "wait",
                    "selector": "#mw-content-text",
                    "timeout": 10000
                },
                {
                    "type": "generate_markdown"
                }
            ]
        }
    }

    headers = {
        "x-api-key": GAFFA_API_KEY,
        "Content-Type": "application/json"
    }

    print("Calling Gaffa API to generate markdown...")
    response = requests.post(
        "https://api.gaffa.dev/v1/browser/requests",
        json=payload,
        headers=headers
    )
    response.raise_for_status()

    markdown_url = response.json()["data"]["actions"][1]["output"]
    print(f"Fetching markdown from: {markdown_url}")

    markdown_response = requests.get(markdown_url)
    markdown_response.raise_for_status()

    return markdown_response.text

print("✅ fetch_markdown_with_gaffa function defined.")

✅ fetch_markdown_with_gaffa function defined.


## Step 4: Fetch the Wikipedia Page

Let's scrape the Wikipedia page on Artificial Intelligence and convert it to Markdown.

In [74]:
url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
markdown = fetch_markdown_with_gaffa(url)

print("✅ Markdown successfully retrieved.\n")
print(markdown[:500])

Calling Gaffa API to generate markdown...
Fetching markdown from: https://storage.gaffa.dev/brq/md/brq_VgjqqFYcrMZ5orY1fhRRS8F6o6MHyN/act_VgjqqBYXR9L3vbe4g4v9KtvusChWgd.md
✅ Markdown successfully retrieved.

Jump to content 
Main menu 
Main menu
 move to sidebar hide
									
Navigation
										
- [Main page](/wiki/Main_Page "Visit the main page [alt-z]")
- [Contents](/wiki/Wikipedia:Contents "Guides to browsing Wikipedia")
- [Current events](/wiki/Portal:Current_events "Articles related to current events")
- [Random article](/wiki/Special:Random "Visit a randomly selected article [alt-x]")
- [About Wikipedia](/wiki/Wikipedia:About "Learn about Wikipedia and how it works")
- [Contact 


## Step 5: Ask Questions with gpt-4o

Now we'll send the Markdown content to gpt-4o's LLM and ask it questions about the page.

In [ ]:
def ask_question(markdown, question):
    client = OpenAI(api_key=OPENAI_API_KEY)

    # Model options (pick one):
    # "gpt-3.5-turbo-16k"  — 16k context window (~12k tokens for content)
    # "gpt-4-turbo"        — 128k context window (recommended for large pages)
    # "gpt-4o"             — 128k context window (faster & cheaper than gpt-4-turbo)
    MODEL = "gpt-4o"

    system_prompt = (
        "You are an assistant helping analyze webpage content. "
        "Answer ONLY using the information provided in the markdown content given by the user. "
        "If the answer is not found in the content, say: "
        "'I could not find that information in the provided content.' "
        "Do not use any outside knowledge or make anything up."
    )

    user_prompt = (
        f"Markdown content:\n{markdown[41170:182645]}\n\n" # Adding only the section of the markdown that contains the meaningful content
        f"Question: {question}\n"
        f"Answer as clearly as possible."
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

print("✅ ask_question function defined.")

✅ ask_question function defined.


## Step 6: Ask Your First Question

Let's ask the LLM a question about the content we just scraped.

In [78]:
question = "What is artificial intelligence and what are its main goals?"
answer = ask_question(markdown, question)

print(f"Question: {question}\n")
print(f"Answer: {answer}")

Question: What is artificial intelligence and what are its main goals?

Answer: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software enabling machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.

The main goals of AI include learning, reasoning, knowledge representation, planning, natural language processing, perception, and support for robotics.


In [79]:
questions = [
    "What is artificial intelligence and what are its main goals?",
    "What are the main risks of artificial intelligence?",
    "How is AI used in healthcare?",
]

for question in questions:
    print(f"Question: {question}\n")
    answer = ask_question(markdown, question)
    print(f"Answer: {answer}\n")
    print("-" * 50)

Question: What is artificial intelligence and what are its main goals?

Answer: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.

The main goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, perception, and support for robotics. AI researchers aim to simulate and create intelligent systems by focusing on particular traits or capabilities such as step-by-step reasoning, dealing with uncertain or incomplete information, and improving performance on specific tasks automatically.

----------------------------------------